# Conv 컨베이어 속도 vs Pour_Zone_CNT 증가율 상관 분석

지난 24시간 동안 아래 4개 태그를 받아서, **컨베이어가 실제로 돌고 있을 때만**
(`Conv_Brake_Time` != 0)을 대상으로 `Pour_Zone_CNT PV`의 증가율(기울기)이
`Conv_SPD SET_HMI`(속도 설정값) / `Hz_In_Conv`(실제 인버터 주파수)와 상관관계가
있는지 확인한다.

- `Conv_Brake_Time == 0`이면 컨베이어가 멈춘 상태라 카운트가 안 늘어난다 →
  전처리에서 이 구간은 통째로 제외한다.
- `Pour_Zone_CNT PV`는 상한(130~140대)에서 다시 1 근처로 떨어지는 **순환 카운터**다
  (실측). 리셋되는 순간의 음수 diff 를 실제 감소로 잘못 세면 증가율 계산이
  완전히 망가지므로(리셋 포함 시 상관관계가 거의 0으로 사라짐), 리셋 구간은
  증가량 계산에서 제외한다.
- 데이터가 0.5초 간격으로 들어오므로, 증가율은 "0.5초 틱마다 `Pour_Zone_CNT PV`가
  몇 개나 늘었는지"로 정의한다. 한 틱에 여러 개가 몰리면 그만큼 기울기가 높다는
  뜻이다. 틱 단위는 대부분 0, 가끔 1~2라 너무 튀므로, 1분 단위로 합산한 값도
  같이 봐서 더 안정적인 비교를 한다.

순서: 1) 추출 → 2) 전처리(정지구간 제외) → 3) 저장 → 4) 증가율 계산(리셋 제외) → 5) 상관분석/시각화

## 1. 모듈 로드 및 파라미터

In [ ]:
import os
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
%matplotlib inline
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

# 이 노트북은 parkkt/conv/ 에 있고, Log_Extractor.py 와 .env 는 레포 루트에 있다.
ROOT_DIR = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.insert(0, ROOT_DIR)

from Log_Extractor import LogExtractor

ENV_PATH = os.path.join(ROOT_DIR, ".env")
SAVE_DIR = os.path.join(os.getcwd(), "extracted_csv")

pd.set_option("display.width", 200)
print(f"ROOT_DIR = {ROOT_DIR}")
print(f"ENV_PATH = {ENV_PATH}  (존재={os.path.exists(ENV_PATH)})")
print(f"SAVE_DIR = {SAVE_DIR}")

In [ ]:
TAGS = [
    "Conv_Brake_Time",
    "Conv_SPD SET_HMI",
    "Pour_Zone_CNT PV",
    "Hz_In_Conv",
]

START_TIME = "-24h"
END_TIME = "now()"
FILE_PREFIX = "Conv_Speed_Corr"

TICK_SEC = 0.5   # PLC 스캔 주기 — 원본 데이터가 이 간격으로 찍힌다
AGG_SEC = 60     # 틱 단위 증가량이 너무 튀어서 같이 볼 1분 집계 구간

## 2. 추출 실행

In [ ]:
extractor = LogExtractor(env_path=ENV_PATH)
raw = extractor.get_data(start_time=START_TIME, end_time=END_TIME, target_tags=TAGS)
raw.shape

## 3. 전처리 — 정지 구간(`Conv_Brake_Time == 0`) 제외

원본은 값이 바뀔 때만 찍히는 이벤트성 로그라 시간 간격이 고르지 않다. 0.5초
간격의 규칙적인 격자(grid)로 먼저 맞춘 뒤(ffill), `Conv_Brake_Time` 이 0인 틱을
찾아 가동 여부를 표시한다.

In [ ]:
grid_index = pd.date_range(raw.index.min(), raw.index.max(), freq=f"{int(TICK_SEC * 1000)}ms")
g = raw.reindex(raw.index.union(grid_index)).sort_index().ffill().reindex(grid_index)
g.index.name = "Time"

running = g["Conv_Brake_Time"] != 0
print(f"전체 틱 {len(g):,}개 중 가동중 {running.sum():,}개 ({running.mean() * 100:.1f}%)")

In [ ]:
# Pour_Zone_CNT PV 는 상한 근처(130~140대)에서 1~수 로 뚝 떨어지는 순환 카운터다
# (실측: 리셋 시 diff 가 -114 ~ -140). 음수 diff 는 실제 감소가 아니라 리셋이므로
# 증가량 계산에서 빼야 한다 — 안 빼면 리셋 한 번이 정상 증가 수백 틱 분을 상쇄해
# 상관관계가 통째로 사라진다(실측: 리셋 포함 Pearson r≈0 → 제외 후 r≈0.7).
# 정지->가동 경계에서도 diff 가 튀지 않도록, 앞뒤 틱이 둘 다 가동중일 때만 인정한다.
prev_running = running.shift(1, fill_value=False)
raw_diff = g["Pour_Zone_CNT PV"].diff()
reset = raw_diff < 0
print(f"Pour_Zone_CNT PV 리셋(음수 diff) {int(reset.sum()):,}회 — 중앙값 {raw_diff[reset].median():.0f} (카운터가 상한에서 순환)")

valid = running & prev_running & (raw_diff >= 0)
diff_cnt = raw_diff.where(valid)
g["Pour_Zone_CNT_증가"] = diff_cnt

## 4. 저장

In [ ]:
if raw.empty:
    print("⚠️ 저장할 데이터가 없습니다.")
else:
    os.makedirs(SAVE_DIR, exist_ok=True)
    timestamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")

    raw_path = os.path.join(SAVE_DIR, f"{FILE_PREFIX}_raw_{timestamp}.csv")
    raw.to_csv(raw_path, encoding="utf-8-sig")

    prep_path = os.path.join(SAVE_DIR, f"{FILE_PREFIX}_전처리_{timestamp}.csv")
    g[running].to_csv(prep_path, encoding="utf-8-sig")

    print(f"💾 원본: {raw_path}")
    print(f"💾 전처리(가동중만, 0.5초 격자): {prep_path}")

## 5. 증가율(기울기) 계산

- **틱 단위**: 0.5초마다 `Pour_Zone_CNT PV`가 몇 개 늘었는지를 그대로 시간당 값으로
  환산한다(`증가_틱당 * 3600 / 0.5`). 값 대부분이 0이고 가끔 1~2라 분포가 매우
  튄다.
- **1분 집계**: 1분 창에 몰린 증가량을 합산해 같은 방식으로 시간당 값으로 환산한다.
  덜 튀어서 상관관계를 보기에 더 안정적이다. 그 창의 90% 이상이 가동중일 때만 쓴다.

In [ ]:
df = g[running].copy()
df["증가_틱당"] = diff_cnt[running]
df["증가율_시간당_틱"] = df["증가_틱당"] * (3600 / TICK_SEC)

df[["Pour_Zone_CNT PV", "증가_틱당", "증가율_시간당_틱", "Conv_SPD SET_HMI", "Hz_In_Conv"]].describe()

In [ ]:
agg = pd.DataFrame({
    "증가_합": diff_cnt.resample(f"{AGG_SEC}s").sum(min_count=1),
    "가동비율": running.resample(f"{AGG_SEC}s").mean(),
    "Conv_SPD SET_HMI": g["Conv_SPD SET_HMI"].resample(f"{AGG_SEC}s").mean(),
    "Hz_In_Conv": g["Hz_In_Conv"].resample(f"{AGG_SEC}s").mean(),
})
agg = agg[agg["가동비율"] > 0.9]
agg["증가율_시간당"] = agg["증가_합"] * (3600 / AGG_SEC)
agg = agg.dropna(subset=["증가율_시간당", "Conv_SPD SET_HMI", "Hz_In_Conv"])
agg.shape

## 6. 상관관계 분석

In [ ]:
def report_corr(x, y, label_x, label_y):
    m = x.notna() & y.notna()
    if m.sum() < 3:
        print(f"{label_x} vs {label_y}: 표본 부족")
        return
    r, p = stats.pearsonr(x[m], y[m])
    rho, sp = stats.spearmanr(x[m], y[m])
    print(f"{label_x} vs {label_y} (n={m.sum():,}) — Pearson r={r:.3f}(p={p:.1g})  Spearman ρ={rho:.3f}(p={sp:.1g})")


print("--- 틱 단위(0.5초) ---")
report_corr(df["증가율_시간당_틱"], df["Conv_SPD SET_HMI"], "증가율(틱)", "Conv_SPD SET_HMI")
report_corr(df["증가율_시간당_틱"], df["Hz_In_Conv"], "증가율(틱)", "Hz_In_Conv")

print("\n--- 1분 집계 ---")
report_corr(agg["증가율_시간당"], agg["Conv_SPD SET_HMI"], "증가율(1분)", "Conv_SPD SET_HMI")
report_corr(agg["증가율_시간당"], agg["Hz_In_Conv"], "증가율(1분)", "Hz_In_Conv")

## 7. 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

pairs = [
    (df["Conv_SPD SET_HMI"], df["증가율_시간당_틱"], "Conv_SPD SET_HMI", "증가율(틱, /시간)", axes[0, 0]),
    (df["Hz_In_Conv"], df["증가율_시간당_틱"], "Hz_In_Conv", "증가율(틱, /시간)", axes[0, 1]),
    (agg["Conv_SPD SET_HMI"], agg["증가율_시간당"], "Conv_SPD SET_HMI", "증가율(1분, /시간)", axes[1, 0]),
    (agg["Hz_In_Conv"], agg["증가율_시간당"], "Hz_In_Conv", "증가율(1분, /시간)", axes[1, 1]),
]

for x, y, xl, yl, ax in pairs:
    m = x.notna() & y.notna()
    ax.scatter(x[m], y[m], s=8, alpha=0.3)
    if m.sum() > 2:
        b, a = np.polyfit(x[m], y[m], 1)
        xs = np.linspace(x[m].min(), x[m].max(), 50)
        ax.plot(xs, a + b * xs, color="red", linewidth=1.5)
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)
    ax.grid(alpha=0.3)

fig.tight_layout()

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 4))
ax1.plot(agg.index, agg["증가율_시간당"], color="steelblue", label="Pour_Zone_CNT 증가율(/시간)")
ax1.set_ylabel("증가율(/시간)", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")

ax2 = ax1.twinx()
ax2.plot(agg.index, agg["Conv_SPD SET_HMI"], color="orange", alpha=0.7, label="Conv_SPD SET_HMI")
ax2.plot(agg.index, agg["Hz_In_Conv"], color="green", alpha=0.7, label="Hz_In_Conv")
ax2.set_ylabel("속도", color="black")

fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.95))
ax1.grid(alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()